# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruzaki11/Flyrank-ml-intern-tasks/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question**

Can observable content and search-performance signals be used to identify and rank pages that are candidates for content refresh, and can a machine-learning model improve upon a simple rule-based baseline?

**Decision it supports**

The analysis supports the decision of which content pages should be prioritized for refresh review. The resulting ranking can help a content team decide which pages to investigate first, rather than treating every page as equally important.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

The analysis uses the provided anonymized content-level starter table. No external datasets or private client-level sources were joined to the analysis.

In [2]:
# these are the data windows
'''
90-day:
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d

30-day:
impressions_last_30d
clicks_last_30d
sessions_last_30d

previous 30-day:
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
'''

'\n90-day:\nimpressions_90d\nclicks_90d\npageviews_90d\nsessions_90d\nusers_90d\nengaged_sessions_90d\nai_sessions_90d\nscroll_events_90d\n\n30-day:\nimpressions_last_30d\nclicks_last_30d\nsessions_last_30d\n\nprevious 30-day:\nimpressions_prev_30d\nclicks_prev_30d\nsessions_prev_30d\n'

Features describe historical search and engagement behavior over 90-day and 30-day windows, including current and previous 30-day periods. Content freshness is represented using fields such as content age and days since the last update. The dataset does not provide a single explicit prediction timestamp for each row, so the analysis treats the supplied historical windows as the available observation period rather than claiming a specific calendar date range.

**The excluded fearures:**

1-client_id

2-content_id

Identifiers were excluded because they identify records or clients rather than representing generalizable predictive signals. client_id is retained only for grouped validation.

3- is_initial_refresh_candidate

This is the prediction target and therefore cannot be included as an input feature

4- ( needs_indexing

is_quick_win

needs_ctr_fix

needs_engagement_fix

is_underperformer

is_declining

health_score

ai_opportunity)

These fields are derived flags or scores related to content recommendations and refresh decisions. Including them could give the model information derived from the target or from existing product rules rather than requiring it to learn from the underlying signals.

5-(future_clicks

future_ctr

next_30d_sessions)

These fields describe outcomes after the prediction point and therefore would not be available when making a real refresh-prioritization decision.



## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions**

The analysis treats is_initial_refresh_candidate as the operational definition of refresh candidacy provided in the FlyRank dataset. The objective is predictive prioritization rather than causal inference: a high predicted probability indicates that a page resembles pages labeled as refresh candidates, not that refreshing the page will necessarily improve its future performance.

Features are assumed to represent information available at or before the prediction point. Historical 30-day and 90-day performance windows are therefore treated as observational inputs, while explicitly identified future-window fields are excluded.
****

**Features**

The model uses observable content, search-performance, engagement, freshness, and trend signals. Numeric features include measures such as search volume, CPC, content age, historical impressions, clicks, sessions, CTR, average position, engagement rate, scroll rate, AI-traffic percentage, and trend percentage. Categorical features include content type, search intent, age/freshness tiers, position and impression tiers, and trend direction.

Missing numeric values are handled using median imputation, while missing categorical values are handled using the most-frequent category. Categorical variables are one-hot encoded, and numeric variables are standardized for models that require scaling. Preprocessing is fitted on the training data through a scikit-learn pipeline to avoid using test-set information during preprocessing.

****

**Label definition**

The prediction target is (is_initial_refresh_candidate), a binary field indicating whether the content record is identified as an initial refresh candidate in the supplied dataset. The target is used only as the outcome variable and is excluded from the feature matrix.
****

**Baseline**

The baseline combines (days_since_last_update) with (trend_direction) to assign a baseline score, reason code, and recommended action. The purpose of the baseline is to provide a transparent and interpretable benchmark that the machine-learning models must improve upon rather than evaluating model performance in isolation.
****

**Validation design**

Because multiple content records can belong to the same client, the model evaluation uses client-grouped validation. client_id is used as the grouping variable so that records from the same client are not intentionally distributed across the training and test sets. Stratified grouped splitting is used to preserve the binary label distribution as far as possible while maintaining client-level separation.

The model and baseline are evaluated on the same held-out records and using the same F1 metric. This makes the comparison between the rule-based baseline and the machine-learning model directly comparable.
****

**Leakage checks**

A dedicated leakage audit was performed before modeling. Fields derived from the target, existing recommendation/product flags, and future outcomes were identified and excluded from the predictive feature set. In particular, is_initial_refresh_candidate itself was excluded, along with label-derived fields such as needs_indexing, is_quick_win, needs_ctr_fix, needs_engagement_fix, is_underperformer, is_declining, health_score, and ai_opportunity. Future-window fields such as future_clicks, future_ctr, and next_30d_sessions were also excluded because they represent information that would occur after the prediction point.

Identifiers such as content_id and client_id were excluded from the model features. client_id was retained only as the grouping variable for validation.

Preprocessing was performed within the training pipeline so that imputers, encoders, and scalers were fitted using training data rather than the complete datase

****

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

| Method               | Validation split        |          F1 score |
| -------------------- | ----------------------- | ----------------: |
| Week-4 rule baseline | Client-grouped test set | **[baseline F1]** |
| Logistic Regression  | Same test set           |        **0.7193** |
| Random Forest        | Same test set           |        **0.9994** |
| Gradient Boosting    | Same test set           | **0.9884** |


## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.